# 07a — Overfitting, Underfitting & Bias-Variance Tradeoff

Corresponds to: Note 07 — Overfitting, Underfitting & Bias-Variance Tradeoff

**Libraries introduced in this notebook:** No new libraries — numpy only (introduced in Note 03).

**What this notebook demonstrates:**
1. L2 regularisation — effect on weight updates vs unregularised
2. Dropout simulation — inverted dropout mask and scaling
3. Learning curves — simulated training and validation loss showing underfitting, overfitting, good fit
4. Early stopping — monitoring validation loss and stopping at the best epoch

---
## Part 1: L2 Regularisation

L2 adds a penalty λ × Σwᵢ² to the loss. The gradient becomes ∂L_orig/∂w + 2λw.

We reproduce the Q10 calculation from Exercise 07 exactly, then compare regularised vs unregularised updates across several steps to show weight decay in action.

In [ ]:
import numpy as np

def sigmoid(z):       return 1 / (1 + np.exp(-z))
def sigmoid_d(z):     s = sigmoid(z); return s * (1 - s)

# Exercise 07 Q10 numbers
w     = 0.8
b     = 0.0
x     = 1.0
y     = 1.0
lam   = 0.5
eta   = 0.5

z      = w * x + b
y_hat  = sigmoid(z)
L_orig = (y - y_hat) ** 2
L_reg  = lam * w**2
L_total = L_orig + L_reg

print('Forward pass (Exercise 07 Q10):')
print(f'  z       = {z:.4f}  (note: 0.8)')
print(f'  y_hat   = {y_hat:.4f}  (note: 0.6900)')
print(f'  L_orig  = {L_orig:.4f}  (note: 0.0961)')
print(f'  L_reg   = {L_reg:.4f}  (note: 0.3200)')
print(f'  L_total = {L_total:.4f}  (note: 0.4161)')

dL_dyhat      = -2 * (y - y_hat)
dL_dz         = dL_dyhat * sigmoid_d(z)
dL_dw_orig    = dL_dz * x
dL_dw_reg     = 2 * lam * w
dL_dw_total   = dL_dw_orig + dL_dw_reg

w_new_reg   = w - eta * dL_dw_total
w_new_unreg = w - eta * dL_dw_orig

print()
print('Gradients:')
print(f'  dL/dw (orig)  = {dL_dw_orig:.4f}  (note: -0.1326)')
print(f'  dL/dw (reg)   = {dL_dw_reg:.4f}  (note: 0.8000)')
print(f'  dL/dw (total) = {dL_dw_total:.4f}  (note: 0.6674)')
print()
print('Weight updates:')
print(f'  w_new with L2    = {w_new_reg:.4f}  (note: 0.4663)')
print(f'  w_new without L2 = {w_new_unreg:.4f}  (note: 0.8663)')
print(f'  L2 pulled weight down by {w_new_unreg - w_new_reg:.4f}')

Now run 20 gradient steps on the same problem to show weight decay over time.

In [ ]:
print('Weight trajectory over 20 steps (same x=1, y=1 each step):')
print(f'  {"Step":>5}   {"w (L2)": >10}   {"w (no L2)": >12}')
print(f'  {"-"*5}   {"-"*10}   {"-"*12}')

w_reg   = 0.8
w_unreg = 0.8

for step in range(20):
    # regularised
    z = w_reg * x + b
    y_hat = sigmoid(z)
    g_orig = -2*(y - y_hat) * sigmoid_d(z) * x
    g_total = g_orig + 2*lam*w_reg
    w_reg -= eta * g_total

    # unregularised
    z2 = w_unreg * x + b
    y_hat2 = sigmoid(z2)
    g2 = -2*(y - y_hat2) * sigmoid_d(z2) * x
    w_unreg -= eta * g2

    if step < 5 or step % 5 == 4:
        print(f'  {step+1:>5}   {w_reg:>10.4f}   {w_unreg:>12.4f}')

print()
print('L2 converges to a smaller weight — regularisation prevents large weights.')

---
## Part 2: Dropout Simulation (Inverted Dropout)

Inverted dropout: during training, each neuron is kept with probability (1−p) and its output is scaled up by 1/(1−p). At test time, all neurons are active with no scaling.

We simulate one hidden layer of 8 neurons across 5 training steps, then show the test-time pass.

In [ ]:
np.random.seed(42)

h = np.array([0.8, 0.6, 0.9, 0.3, 0.7, 0.5, 0.4, 0.2])   # hidden activations
p = 0.5   # dropout rate

print(f'Hidden activations: {h}')
print(f'Dropout rate p = {p}  (keep probability = {1-p})')
print(f'Inverted dropout scale factor = 1/(1-p) = {1/(1-p):.1f}')
print()
print('Training steps (different mask each step):')
print(f'  {"Step":>5}   {"Mask":>20}   {"Active neurons":>15}   {"Mean output":>12}')
print(f'  {"-"*5}   {"-"*20}   {"-"*15}   {"-"*12}')

for step in range(5):
    mask = (np.random.rand(len(h)) > p).astype(float)
    h_dropped = h * mask / (1 - p)   # inverted dropout
    active = int(mask.sum())
    print(f'  {step+1:>5}   {mask.astype(int)}   {active:>15}   {h_dropped.mean():>12.4f}')

print()
print('Test time (no dropout, no scaling):')
print(f'  All neurons active: {h}')
print(f'  Mean output: {h.mean():.4f}')
print()
print('Expected output magnitude is consistent between training and test.')

---
## Part 3: Learning Curves

We simulate training and validation loss over 60 epochs for three scenarios: underfitting, overfitting, and good fit.

These are the same curves shown in the note image — here we generate the numbers explicitly.

In [ ]:
epochs = np.arange(1, 61)

# Underfitting: both losses high and flat
train_under = 0.85 * np.exp(-epochs / 200) + 0.72
val_under   = 0.85 * np.exp(-epochs / 200) + 0.75

# Overfitting: training falls, validation rises after epoch 15
train_over  = 0.9 * np.exp(-epochs / 18) + 0.05
val_over    = 0.9 * np.exp(-epochs / 40) + 0.05 + np.maximum(0, (epochs - 15) * 0.012)

# Good fit: both decrease and converge
train_good  = 0.85 * np.exp(-epochs / 15) + 0.08
val_good    = 0.85 * np.exp(-epochs / 15) + 0.14

print('Learning curve snapshots at epochs 1, 10, 30, 60:')
print(f'  {"Epoch":>6}   {"Under train":>12}   {"Under val":>10}   {"Over train":>11}   {"Over val":>9}   {"Good train":>11}   {"Good val":>9}')
print(f'  {"-"*6}   {"-"*12}   {"-"*10}   {"-"*11}   {"-"*9}   {"-"*11}   {"-"*9}')
for i in [0, 9, 29, 59]:
    print(f'  {epochs[i]:>6}   {train_under[i]:>12.4f}   {val_under[i]:>10.4f}   {train_over[i]:>11.4f}   {val_over[i]:>9.4f}   {train_good[i]:>11.4f}   {val_good[i]:>9.4f}')

print()
print('Underfitting: both losses high throughout — gap is small')
print(f'  Final gap (val - train): {val_under[-1] - train_under[-1]:.4f}')
print()
print('Overfitting: training loss low, validation loss high — large gap')
print(f'  Final train loss: {train_over[-1]:.4f},  val loss: {val_over[-1]:.4f},  gap: {val_over[-1] - train_over[-1]:.4f}')
print()
print('Good fit: both losses low, small gap')
print(f'  Final train loss: {train_good[-1]:.4f},  val loss: {val_good[-1]:.4f},  gap: {val_good[-1] - train_good[-1]:.4f}')

---
## Part 4: Early Stopping

We simulate the overfitting scenario and implement early stopping with a patience window.

The model is saved at the epoch with the lowest validation loss. Training stops if validation loss does not improve for `patience` epochs.

In [ ]:
patience = 10
best_val  = float('inf')
best_epoch = 0
wait = 0
stopped_at = None

print(f'Early stopping with patience = {patience}:')
print(f'  {"Epoch":>6}   {"Train loss":>11}   {"Val loss":>9}   {"Best val":>9}   {"Wait":>5}')
print(f'  {"-"*6}   {"-"*11}   {"-"*9}   {"-"*9}   {"-"*5}')

for i, ep in enumerate(epochs):
    tl = train_over[i]
    vl = val_over[i]

    if vl < best_val:
        best_val   = vl
        best_epoch = ep
        wait = 0
    else:
        wait += 1

    if ep <= 5 or ep % 10 == 0 or wait == patience:
        print(f'  {ep:>6}   {tl:>11.4f}   {vl:>9.4f}   {best_val:>9.4f}   {wait:>5}')

    if wait == patience:
        stopped_at = ep
        break

print()
print(f'Stopped at epoch {stopped_at}.')
print(f'Best model saved at epoch {best_epoch} with validation loss {best_val:.4f}.')
print(f'Training without early stopping would continue to epoch 60 — overfitting.')

---
## Summary

| What was demonstrated | Key result |
|---|---|
| L2 regularisation | Gradient = original gradient + 2λw — weight pulled toward zero each step |
| L2 vs unregularised | L2 converges to smaller weight; unregularised converges to larger weight |
| Inverted dropout | Scale up by 1/(1−p) during training; no scaling at test time — magnitudes consistent |
| Learning curves | Underfitting: both losses high, small gap; Overfitting: large gap; Good fit: both low, small gap |
| Early stopping | Stops training when validation loss stops improving — saves best model before overfitting |